In [1]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")


[nltk_data] Downloading package stopwords to C:\Users\Nossim
[nltk_data]     Sankaire\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:

data_path = "../data/train_raw.csv"
df = pd.read_csv(data_path, encoding="utf-8", on_bad_lines="skip")

print("Dataset loaded:", df.shape)
print("Columns available:", df.columns.tolist())

df.head(10)


Dataset loaded: (400, 12)
Columns available: ['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI', 'DDX SNOMED']


,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...
2,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder)\n25458004 |...
3,ID_QOQTK,Uasin Gishu,National Referral Hospitals,12.0,I am a nurse with 12 years of experience in Pr...,Critical Care,INTERNAL MEDICINE,SUMMARY\n\n72-year-old female with inability t...,"Given ER's clinical presentation and vitals, t...",to me with this query. Based on the informatio...,This 92-year-old female patient (ER) presents ...,14760008 | Constipation (finding)\n419284004 |...
4,ID_ZFJBM,Uasin Gishu,National Referral Hospitals,16.0,I am a nurse with 16 years of experience in Ge...,Adult Health,INTERNAL MEDICINE,"A 22 year old female presents with headache, d...",The 22-year-old female patient is presenting w...,Thank you for presenting this case. Based on t...,This 22-year-old female patient presents with ...,95874006 | Carbon monoxide poisoning from fire...
5,ID_SFQGM,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,Maternal and Child Health,OBSTETRICS AND GYNAECOLOGY,Summary\nA 40 year old para 3 + 0 G4 at 38 wee...,Managing the care of a newborn born to a mothe...,"As a community nurse in Kenya, it's great that...",Managing a baby born to a mother on intensive ...,267317007 | Tuberculosis in pregnancy (disorde...
6,ID_ABUJJ,Uasin Gishu,National Referral Hospitals,20.0,I am a nurse with 20 years of experience in Ge...,General Emergency,SURGERY,"SUMMARY\nA 33-year-old male with tooth pain, b...",Given the patient's history of a motorbike acc...,"Based on the patient's presentation, I would p...",The priority management of this patient involv...,718080002 | Gingival excess (finding)\n3262000...
7,ID_SUOCB,Uasin Gishu,National Referral Hospitals,NaN,I am a nurse working in a National Referral Ho...,Child Health,SURGERY,A 10 year old presenting with lump fever and l...,The presentation of a 10-year-old boy with a l...,"Based on the symptoms and X-ray findings, I'll...",Given the presentation of a 10-year-old boy wi...,307576001 | Osteosarcoma of bone (disorder)\n6...
8,ID_MIAYN,Uasin Gishu,National Referral Hospitals,16.0,I am a nurse with 16 years of experience in Ge...,General Emergency,CRITICAL CARE,"Summary\n21 year old male with chest pain, tig...",In the case of a 21-year-old male patient with...,"Given the patient's symptoms and presentation,...",This 21-year-old male presents with a severe a...,195967001 | Asthma (disorder)\n233604007 | Pne...
9,ID_OHZDT,Kakamega,Sub-county Hospitals and Nursing Homes,8.0,I am a nurse with 8 years of experience in Gen...,General Emergency,CRITICAL CARE,Summary\n\nA patient with burns on chest and f...,"Based on the information you've provided, the ...",Given the patient's presentation with burns on...,The patient's difficulty breathing

In [3]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("_x0001_", " ")
    text = re.sub(r"\s+", " ", text)   # normalize spaces
    return text.strip()

df["Prompt_clean"] = df["Prompt"].apply(clean_text)

df[["Prompt", "Prompt_clean"]].head(5)


,Prompt,Prompt_clean
0,I am a nurse with 18 years of experience in Ge...,I am a nurse with 18 years of experience in Ge...
1,I am a nurse with 17 years of experience in Ge...,I am a nurse with 17 years of experience in Ge...
2,I am a nurse with 12 years of experience in Ge...,I am a nurse with 12 years of experience in Ge...
3,I am a nurse with 12 years of experience in Pr...,I am a nurse with 12 years of experience in Pr...
4,I am a nurse with 16 years of experience in Ge...,I am a nurse with 16 years of experience in Ge...


In [4]:
def segment_case(text):
    if pd.isna(text):
        return pd.Series(["", "N/A", "N/A"])
    
    text = str(text).strip()
    symptoms, diagnosis, plan = text, "N/A", "N/A"

    # split narrative vs questions
    if "questions" in text.lower():
        parts = re.split(r"questions[:\-]?", text, flags=re.IGNORECASE, maxsplit=1)
        symptoms = parts[0].strip()
        questions = parts[1].strip() if len(parts) > 1 else ""
    else:
        if "?" in text:
            parts = text.split("?", 1)
            symptoms = (parts[0] + "?").strip()
            questions = parts[1]
        else:
            questions = ""

    # classify questions
    diagnosis_q, plan_q = [], []
    for q in re.split(r"\?+", questions):
        q = q.strip()
        if not q:
            continue
        ql = q.lower()
        if any(kw in ql for kw in ["diagnosis", "dx", "cause", "confirm", "differential"]):
            diagnosis_q.append(q)
        elif any(kw in ql for kw in ["management", "treatment", "care", "plan", "action", "protocol", "investigation", "what should i do"]):
            plan_q.append(q)
        else:
            plan_q.append(q)

    diagnosis = " | ".join(diagnosis_q) if diagnosis_q else "N/A"
    plan = " | ".join(plan_q) if plan_q else "N/A"

    return pd.Series([symptoms, diagnosis, plan])
df[["Symptoms", "Diagnosis_Qs", "Plan_Qs"]] = df["Prompt_clean"].apply(segment_case)
df[["Prompt_clean", "Symptoms", "Diagnosis_Qs", "Plan_Qs"]].head(10)


,Prompt_clean,Symptoms,Diagnosis_Qs,Plan_Qs
0,I am a nurse with 18 years of experience in Ge...,I am a nurse with 18 years of experience in Ge...,N/A,1. What is the immediate treatment protocol fo...
1,I am a nurse with 17 years of experience in Ge...,I am a nurse with 17 years of experience in Ge...,What is the diagnosis of the patient,What is the most immediate management | What h...
2,I am a nurse with 12 years of experience in Ge...,I am a nurse with 12 years of experience in Ge...,N/A,"I had were, an analgesic first | or this patie..."
3,I am a nurse with 12 years of experience in Pr...,I am a nurse with 12 years of experience in Pr...,What is the diagnosis,What emergency care should patient ER receive ...
4,I am a nurse with 16 years of experience in Ge...,I am a nurse with 16 years of experience in Ge...,N/A,How do I manage the patient | What investigati...
5,I am a nurse with 12 years of experience in Ge...,I am a nurse with 12 years of experience in Ge...,N/A,N/A
6,I am a nurse with 20 years of experience in Ge...,I am a nurse with 20 years of experience in Ge...,N/A,N/A
7,I am a nurse working in a National Referral Ho...,I am a nurse working in a National Referral Ho...,What is the diagnosis,What's the management
8,I am a nurse with 16 years of experience in Ge...,I am a nurse with 16 years of experience in Ge...,N/A,N/A
9,I am a nurse with 8 years of experience in Gen...,I am a nurse with 8 years of experience in Gen...,N/A,The patient develops difficulty in breathing w...


In [5]:
output_path = "../data/train_clean.csv"
df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)


Cleaned dataset saved to: ../data/train_clean.csv
